In [4]:
import sys

# Python 3.11+ limits conversion of very large integer strings by default. Flipper's exact computation for this long word can exceed that limit.
if hasattr(sys, 'set_int_max_str_digits'):
    sys.set_int_max_str_digits(0)

import flipper
surface = 'SB_10'
S = flipper.load(surface)
big_word = 's_5.S_6.S_7.s_8.s_8.s_7.s_6.s_5.S_4.S_3.S_1.S_0.S_2.S_1.S_7.s_8.s_8.s_7.s_6.S_5.S_4.S_3.S_2.s_8.s_7.S_6.S_5.S_4.S_3.s_4.s_3.s_2.s_1.s_0.s_0.s_1.s_2.s_3.s_4.s_5.s_6.s_7.s_7.S_6.S_5.S_4.s_8.s_7.s_6.s_5.s_4.s_3.s_2.s_1.s_0.s_0.s_1.s_2.s_3.s_4.s_5.s_6.s_7.s_8.s_8.S_7.S_6.S_5.S_7.s_6.s_5.s_4.s_3.s_2.s_1.s_0.s_0.s_1.s_2.s_3.s_4.s_5.s_6.s_7.s_8.s_8.S_7.S_6.s_7.s_6.s_5.s_4.s_3.s_2.s_1.s_0.s_0.s_1.s_2.s_3.s_4.s_5.s_6.s_7.s_8.s_8.S_7.s_7.s_6.s_5.s_4.s_3.s_2.s_1.s_0.s_0.s_1.s_2.s_3.s_4.s_5.s_6.s_7.s_8.s_8.S_5.s_6.s_7.S_8.S_8.S_7.S_6.S_5.s_4.s_3.s_1.s_0.s_2.s_1.s_7.S_8.S_8.S_7.S_6.s_5.s_4.s_3.s_2.S_8.S_7.s_6.s_5.s_4.s_3.S_4.S_3.S_2.S_1.S_0.S_0.S_1.S_2.S_3.S_4.S_5.S_6.S_7.S_7.s_6.s_5.s_4.S_8.S_7.S_6.S_5.S_4.S_3.S_2.S_1.S_0.S_0.S_1.S_2.S_3.S_4.S_5.S_6.S_7.S_8.S_8.s_7.s_6.s_5.s_7.S_6.S_5.S_4.S_3.S_2.S_1.S_0.S_0.S_1.S_2.S_3.S_4.S_5.S_6.S_7.S_8.S_8.s_7.s_6.S_7.S_6.S_5.S_4.S_3.S_2.S_1.S_0.S_0.S_1.S_2.S_3.S_4.S_5.S_6.S_7.S_8.S_8.s_7.S_7.S_6.S_5.S_4.S_3.S_2.S_1.S_0.S_0.S_1.S_2.S_3.S_4.S_5.S_6.S_7.S_8.S_8'
h = S.mapping_class(big_word)
print('h is %s.' % h.nielsen_thurston_type())

h is Pseudo-Anosov.


In [5]:
# Exact algebraic certificate for the pseudo-Anosov dilatation.
lam = h.dilatation()
pf = lam.minpoly()
I = lam.interval(accuracy=40)

print('exact minimal polynomial:', pf)
print('certified interval:', I)

# Verify in Sage that the displayed polynomial is irreducible, isolate all of its positive real roots, and identify the largest.
R.<t> = PolynomialRing(QQ)
degree = int(pf.poldegree())
p = R([ZZ(pf.polcoef(i)) for i in range(degree + 1)])
a = QQ(I.lower) / 10**I.precision
b = QQ(I.upper) / 10**I.precision

assert p.is_irreducible()
assert p.number_of_roots_in_interval(a, b) == 1
assert p(a) < 0 < p(b)
positive_roots = sorted(r for r in p.roots(AA, multiplicities=False) if r > 0)
displayed_lower = QQ(819981448) / 10**8
displayed_upper = QQ(819981449) / 10**8
assert positive_roots and all(r < displayed_lower for r in positive_roots[:-1])
assert a < positive_roots[-1] < b
assert displayed_lower < positive_roots[-1] < displayed_upper
print('isolated positive real roots:', positive_roots)
print('verified: lambda is the largest positive real root of p and is isolated in [a, b]')

exact minimal polynomial: x^14 - 16*x^13 + 78*x^12 - 108*x^11 - 99*x^10 + 334*x^9 + 56*x^8 - 556*x^7 + 56*x^6 + 334*x^5 - 99*x^4 - 108*x^3 + 78*x^2 - 16*x + 1
certified interval: [8.19981448680031976287227880930332657785974, 8.19981448680031976287227880930332657785975]
isolated positive real roots: [0.12195397854546020?, 0.1992211384532293?, 0.5855377673178387?, 1.707831767335317?, 5.019547663285579?, 8.199814486800320?]
verified: lambda is the largest positive real root of p and is isolated in [a, b]


In [6]:
# Exact train-track certificate. This produces a 42 x 42 integral matrix.
FM = h.hitting_matrix()
M = matrix(ZZ, FM.rows)
assert all(entry >= 0 for entry in M.list())
chi = M.charpoly()

print('train-track matrix dimensions:', M.nrows(), 'x', M.ncols())
print('train-track hitting matrix:')
print(M)
print('characteristic polynomial factorization:', chi.factor())

chi_in_R = R(chi.list())
assert chi_in_R % p == 0
remaining_factor = chi_in_R // p
assert all(factor.is_cyclotomic() for factor, _ in remaining_factor.factor())
print('verified: the dilatation polynomial divides the characteristic polynomial and all remaining factors are cyclotomic')

train-track matrix dimensions: 42 x 42
train-track hitting matrix:
[0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0]
[0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 1 0 0]
[0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 1 0 0 0 0 1 0]
[0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0]
[0 2 0 0 0 2 0 0 1 0 0 0 0 0 2 0 0 0 0 0 0 1 1 0 2 0 1 0 2 1 0 1 1 0 0 0 0 1 0 1 0 0]
[0 2 0 1 0 0 0 0 0 0 0 0 2 0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 1 0 2 0 0 0 0 1 0 0 0 0 1 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 1 1 0 0 0 0 0 0 1 0 0]
[0 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 1 1 0 0 0 1 1 1 0 0 0 0 0 2 0 0]
[1 0 0 0 1 0 1 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 2 0 0 0 0 1 0 0 0 2 1 0 0 0 0 0 0 2 0 0]
[0 0 0 0 0 0 0 0 0 0 2 0 0 0 0 0 0 2 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0]
[0 1 0 1 0 0 0 0 0 0 0 0 1 0 0 1 1 0 0 0 0 0 0 0 0 1 0 1 0 0 1 0 1 0 0 1 